In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("srinathmkce/wiki-ai-filtered", split="train")
dataset

In [ ]:
dataset_dict = dataset.to_pandas()[['id', 'url', 'title']].to_dict(orient='records')

In [ ]:
prompt = f"""
You are a helpful assistant that can answer questions and help with tasks.
Objective is to identify 100 relevant articles related to artificial intelligence.
You are provided with the ID, URL and the title of the wikipedia article.
Analyze the title and determine if it is relevant to artificial intelligence.
Avoid the articles which are list or collection of tools.
Return the ID of the 100 articles that are relevant to artificial intelligence.
Return only the 100 IDs, no other text. Make sure you have 100 IDs.
{dataset_dict}
"""


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,  
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
response = model.invoke(prompt)

In [ ]:
response.content

In [ ]:
article_id_list = response.content.split('\n')
filtered_dataset = dataset.filter(lambda x: x['id'] in article_id_list)
filtered_dataset

In [ ]:
filtered_dataset.to_pandas()

In [ ]:
filtered_dataset.push_to_hub("srinathmkce/wiki-ai-filtered", revision="filtered-100")

In [ ]:
from datasets import load_dataset
filtered_dataset = load_dataset("srinathmkce/wiki-ai-filtered", revision="filtered-100")
filtered_dataset

In [ ]:
df = filtered_dataset['train'].to_pandas()
df

In [ ]:
# check number of words in the text column 
df['len'] = df['text'].apply(lambda x: len(x.split()))
df

In [ ]:

import plotly.express as px
fig = px.bar(df, x=df['id'].astype(str), y='len', labels={'len': 'Number of Words', 'id': 'Article ID'}, title='Number of Words in Each Article')
fig.update_layout(xaxis_tickangle=90)
fig.show()